# M6.A5 — 베이스라인 비교 학습

> 산출 근거: `docs/plan/ai/phase_06_model.md` M6.A5 · 하네스: M6.A4 확정 규칙(`../data_prep/preprocess.py`)
> 입력: keep 20열(`02_features.ipynb` §7) · 작성: 2026-07-27

🔒 **공개 저장소 데이터 정책** — 실매장 매출의 절대 금액(원 단위 수치·그림·표)은 커밋하지 않는다.
본 노트북은 **출력 제거 상태로 추적**되며, 본문 서술은 비율·배수·sMAPE(%)만 사용한다.
절대 MAE 등 전체 수치는 로컬 재실행으로 확인한다 (실행법: `AI/README.md`).

**목적** — 나이브 기준선·통계 모델·GBM을 동일 하네스로 비교해 M6.A6(초기 모델 선정)의 근거를 만든다.
**하네스 (M6.A4 확정)** — 월 단위 walk-forward: 선택 fold 5개(2025-09~12, 2026-03)로 비교·선정하고,
**최종 fold(2026-04)는 test로 봉인**해 승자만 1회 평가. 타깃 log1p, 승자 선정 기준 = 선택 fold 평균 MAE(사전 고정).
지표는 MAE(선정)+sMAPE(보고) — MAPE는 소액일(주문 1건 날) 왜곡으로 제외(EDA §2.4).

## 판정 요약 (TL;DR)

1. **승자: LightGBM 비율 타깃 하이브리드(V1)** — 타깃을 `log1p(y) − log1p(roll7_mean)`(최근 7영업일
   평균 대비 편차)로 바꾼 LGBM. 선택 5-fold에서 **MA-7 대비 MAE -8.8%, naive-요일 대비 skill +23.1%**, sMAPE 49.7%.
2. **경고적 발견** — 순정 GBM(log 타깃)·SARIMA·XGBoost는 전부 **단순 7영업일 이동평균(MA-7)을 못 이겼다**.
   일 매출은 노이즈 지배(모든 모델 sMAPE 49~66%) — 소규모 매장의 일 단위 변동 절반가량은 예측 불가 영역.
3. **하이브리드가 이기는 구조** — 수준(level)은 MA-7이 추적하고 모델은 편차(요일·학사·공휴일·regime)만 학습.
   특히 **재개장 fold(2026-03)에서 MA-7 대비 -27%** — 수준 급변 구간에서 피처 기반 보정이 결정적.
4. **GBM 3사 대결(같은 하이브리드 구조·§3b)** — LightGBM 승. XGBoost-ratio +7.9%, CatBoost-ratio +9.9%(vs V1),
   둘 다 regime fold에서 열세. GBM 계열 선택 논쟁 종결.
5. **hold 피처 최종 처분** — month sin/cos(+1.1%)·술 비중(+4.1%)·gap(+2.3%) 전부 무익 → **keep 20열 확정**.
   is_exam만 학사일정 검수 후 재평가 대상으로 잔류.
6. **봉인 test(2026-04) 1회 개봉** — V1 **sMAPE 30.0%**, MA-7 대비 MAE **-19.6%**, naive-요일 대비 **-54%**. 일반화 확인.
7. **M6.A6 권고** — 주 모델 **LGBM 하이브리드(V1)** + 보조 baseline **MA-7**(운영 fallback·drift 감시).
   SARIMA·XGBoost·CatBoost·순정 LGBM 기각. DNN 전환(M6.A9)은 보류 권고. 비율 타깃 구조는 M6.A7 XAI에도
   유리 — SHAP이 "평소 대비 왜 높/낮은가"를 직접 설명. AutoML(AutoGluon-TS)은 M6.A9 DNN probe에서 활용 예정.

> 재현 노트 — 비율 타깃은 roll7 워밍업 7행의 라벨이 NaN이 된다. **해당 행은 학습에서 명시 제거**가 V1의
> 정의다(라이브러리별 NaN 라벨 처리가 미정의·비호환이라 결과가 달라짐 — XGBoost는 에러, LGBM은 암묵 처리).

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore")

_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
sys.path.insert(0, str(AI_DIR / "data_prep"))
import preprocess as pp  # M6.A4 규칙 SSOT

PAL = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "gray": "#d9d8d4", "ink2": "#52514e"}
_installed = {f.name for f in fm.fontManager.ttflist}
plt.rcParams.update({
    "font.family": [f for f in ("AppleGothic", "Apple SD Gothic Neo", "NanumGothic") if f in _installed] or ["sans-serif"],
    "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100,
})

feat = pd.read_parquet(AI_DIR / "data/processed/features_daily.parquet").set_index("date")
feat, _ = pp.impute_weather(feat)           # M6.A4 §2 — 기상 1건 선형 보간
ob = feat[feat.is_open].copy()
y = ob.total_amount
r7 = ob.roll7_mean                            # 하이브리드의 수준(level) 기준선

folds = pp.make_monthly_folds(ob.index)
SEL, TEST = folds[:-1], folds[-1]             # 마지막 fold(2026-04)는 test 봉인
print("선택 fold:", [f["month"] for f in SEL], "| test(봉인):", TEST["month"])

KEEP = ["is_holiday", "semester_week", "is_semester_first2w", "temp_avg", "temp_range",
        "lag1_sales", "lag1_tx", "lag_dow_sales", "roll7_mean", "roll4dow_mean", "roll7_atv",
        "is_post_renewal", "days_since_reopen"]  # 02_features §7 keep (dow 원핫 별도)

def matrix(extra=()):
    X = ob[KEEP + list(extra)].copy()
    X = pd.concat([X, pd.get_dummies(ob.index.dayofweek, prefix="dow").set_index(ob.index)], axis=1)
    b = X.select_dtypes(bool).columns
    X[b] = X[b].astype(int)
    return X

def mae(a, p): return float(np.mean(np.abs(a - p)))
def smape(a, p): return float(np.mean(2 * np.abs(a - p) / (a + np.abs(p))) * 100)
won = lambda x: f"{x:,.0f}"

LGB_P = dict(n_estimators=600, learning_rate=0.05, num_leaves=15, min_child_samples=10,
             subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=-1)

## §1 후보 정의

| 후보 | 내용 | 역할 |
|---|---|---|
| Naive-1 | 직전 영업일 매출(lag1) | 최소 기준선 |
| Naive-요일 | 전주 같은 요일 매출(lag_dow, 없으면 lag1) | 주간 주기 기준선 · **skill 분모** |
| MA-7 | 직전 7영업일 평균(roll7_mean) | 스무딩 기준선 |
| SARIMA(1,0,1)(1,1,1)₇ | 캘린더 일 단위 log1p, 무매출일 NaN(Kalman 처리), 검증은 one-step-ahead | 통계 모델 대표 |
| LightGBM / XGBoost | keep 20열, log1p 타깃, 스크리닝 설정 | GBM 대표 |
| V1~V4 변형 (§3) | 비율 타깃 `log(y/roll7)`·l1 손실·블렌드 | 하이브리드 후보 |
| XGBoost·CatBoost-ratio (§3b) | 승자와 같은 비율 타깃 구조로 3사 공정 비교 | GBM 계열 확정용 |

평가 방식 주의 — fold의 train은 검증 월 시작 시점에 고정되지만 lag·rolling 피처는 검증 월 내 일별 실측으로
갱신된다(one-step-ahead). 운영의 야간 배치(일 단위 예측·주 단위 재학습)와 정합.

In [ ]:
# §2 기본 6종 — 선택 5-fold 비교
import lightgbm as lgb
import xgboost as xgb_
from statsmodels.tsa.statespace.sarimax import SARIMAX

def pred_naive1(tr, va):   return ob.loc[va, "lag1_sales"]
def pred_naivedow(tr, va): return ob.loc[va, "lag_dow_sales"].fillna(ob.loc[va, "lag1_sales"])
def pred_ma7(tr, va):      return ob.loc[va, "roll7_mean"]

def pred_sarima(tr, va):
    cal_tr = pd.date_range(ob.index.min(), tr.max(), freq="D")
    s_tr = np.log1p(y.reindex(cal_tr))                     # 무매출일 NaN — Kalman이 결측 처리
    cal_va = pd.date_range(tr.max() + pd.Timedelta("1D"), va.max(), freq="D")
    s_va = np.log1p(y.reindex(cal_va))
    r = SARIMAX(s_tr, order=(1, 0, 1), seasonal_order=(1, 1, 1, 7),
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    p = np.expm1(r.extend(s_va).get_prediction(dynamic=False).predicted_mean)
    return p.reindex(va)

def _gbm_factory(kind):
    def f(tr, va):
        X = matrix()
        ytr, yva = np.log1p(y.loc[tr]), np.log1p(y.loc[va])
        if kind == "lgbm":
            m = lgb.LGBMRegressor(**LGB_P)
            m.fit(X.loc[tr], ytr, eval_set=[(X.loc[va], yva)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        else:
            m = xgb_.XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=4,
                                  min_child_weight=5, subsample=0.9, colsample_bytree=0.9,
                                  random_state=42, verbosity=0, early_stopping_rounds=50)
            m.fit(X.loc[tr], ytr, eval_set=[(X.loc[va], yva)], verbose=False)
        return pd.Series(np.expm1(m.predict(X.loc[va])), index=va)
    return f

BASE = {"Naive-1": pred_naive1, "Naive-요일": pred_naivedow, "MA-7": pred_ma7,
        "SARIMA": pred_sarima, "LightGBM(log)": _gbm_factory("lgbm"), "XGBoost(log)": _gbm_factory("xgb")}

def eval_folds(fn, fold_list):
    out = []
    for f in fold_list:
        p = fn(f["train"], f["val"])
        a = y.loc[f["val"]]
        out.append((mae(a, p), smape(a, p)))
    return out

res = {n: eval_folds(fn, SEL) for n, fn in BASE.items()}
base_mae = np.mean([r[0] for r in res["Naive-요일"]])
tbl = pd.DataFrame({n: {"MAE 평균(원)": won(np.mean([r[0] for r in rs])),
                        "sMAPE 평균": f"{np.mean([r[1] for r in rs]):.1f}%",
                        "skill vs Naive-요일": f"{1 - np.mean([r[0] for r in rs]) / base_mae:+.1%}"}
                   for n, rs in res.items()}).T.sort_values("sMAPE 평균")
display(tbl)
fold_tbl = pd.DataFrame({n: [won(r[0]) for r in rs] for n, rs in res.items()},
                        index=[f["month"] for f in SEL]).T
print("fold별 MAE(원):"); display(fold_tbl)

### §2 관찰 — 기본 후보 결과

- **MA-7이 1위** — 순정 LightGBM(-5%p)·XGBoost·SARIMA·나이브 전부를 이겼다. skill 순서:
  MA-7 +15.7% > SARIMA +8.5% > LGBM +5.3% > XGB +2.4% > Naive-1 +1.4% > Naive-요일(기준 0%).
- 해석: 일 매출의 단기 변동(주문 수 자체가 적은 소규모 매장)은 노이즈가 지배적이라, MAE 기준에선
  **스무딩이 점 예측의 강력한 우승 후보**가 된다. 순정 GBM은 lag 피처의 노이즈에 과반응.
- SARIMA는 주간 계절성은 잡지만 결측(휴무) 많은 시계열에서 수준 추적이 둔함. XGBoost는 LGBM과 동급 이하
  — GBM 계열 내 차이는 미미해서 이후 변형 실험은 LightGBM으로만 진행.
- 모든 모델 sMAPE 49~66% — **일 단위 점 예측의 노이즈 한계선**이 높다는 뜻. M6.A8 신뢰도 기준 설계의 근거.

In [ ]:
# §3 하이브리드 변형 — 수준은 MA-7, 편차만 모델이 학습
# 비율 타깃은 roll7 워밍업 행의 라벨이 NaN → 학습에서 명시 제거(V1 정의의 일부)
def v_run(fold_list, ratio=True, objective="l2", blend=0.0, extra=()):
    X = matrix(extra)
    out = []
    for f in fold_list:
        tr, va = f["train"], f["val"]
        if ratio:
            ytr = np.log1p(y.loc[tr]) - np.log1p(r7.loc[tr])
            yva = np.log1p(y.loc[va]) - np.log1p(r7.loc[va])
        else:
            ytr, yva = np.log1p(y.loc[tr]), np.log1p(y.loc[va])
        keep_tr = ytr.notna()
        m = lgb.LGBMRegressor(objective=objective, **LGB_P)
        m.fit(X.loc[tr][keep_tr], ytr[keep_tr], eval_set=[(X.loc[va], yva)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        raw = pd.Series(m.predict(X.loc[va]), index=va)
        p = np.expm1(raw + np.log1p(r7.loc[va])) if ratio else np.expm1(raw)
        if blend:
            p = blend * r7.loc[va] + (1 - blend) * p
        out.append((mae(y.loc[va], p), smape(y.loc[va], p)))
    return out

VAR = {"V1 비율타깃 log(y/roll7)": dict(ratio=True),
       "V2 log(y)+l1 손실": dict(ratio=False, objective="l1"),
       "V3 비율타깃+l1": dict(ratio=True, objective="l1"),
       "V4 블렌드 0.5·MA7+0.5·V3": dict(ratio=True, objective="l1", blend=0.5)}
vres = {n: v_run(SEL, **kw) for n, kw in VAR.items()}
vtbl = pd.DataFrame({n: {"MAE 평균(원)": won(np.mean([r[0] for r in rs])),
                         "sMAPE 평균": f"{np.mean([r[1] for r in rs]):.1f}%",
                         "vs MA-7": f"{np.mean([r[0] for r in rs]) / np.mean([r[0] for r in res['MA-7']]) - 1:+.1%}",
                         "skill vs Naive-요일": f"{1 - np.mean([r[0] for r in rs]) / base_mae:+.1%}"}
                    for n, rs in vres.items()}).T
display(vtbl)
print("V1 fold별 MAE(원):", [won(r[0]) for r in vres["V1 비율타깃 log(y/roll7)"]])
print("MA-7 fold별 MAE(원):", [won(r[0]) for r in res["MA-7"]])

### §3 관찰 — 하이브리드가 판을 뒤집음

- **V1(비율 타깃, l2)이 전체 1위** — MA-7 대비 **-8.8%**, naive-요일 대비 **+23.1% skill**. V3(l1)·V4(블렌드)도
  MA-7을 이기지만 V1에 못 미침 → 추가 블렌딩·손실 변경 불필요.
- fold별로 보면 V1은 5개 중 4개에서 MA-7과 비기거나 우세하고, 특히 **재개장 fold(2026-03)에서 -27%** —
  개강·재개장으로 수준이 급변하는 구간에서 요일·학사·regime 피처의 보정이 작동한다는 직접 증거.
  유일한 열세 fold(2025-12)는 연말 변동을 편차 모델이 과보정한 구간.
- 구조적 이점: ① 모델이 "평소 대비 몇 %"만 배우므로 regime 간 수준 차이에 강건 ② M6.A7 SHAP이
  편차의 원인(요일·학기·공휴일)을 그대로 설명 — 예측 근거 문구와 1:1 대응 ③ 신규 매장 cold-start에도
  "짧은 이력의 MA + 보정"으로 이식 용이(Phase 7 설계 참고).

In [ ]:
# §3b GBM 3사 공정 비교 — 전부 같은 비율 타깃·NaN 라벨 제거·조기 종료 조건
import xgboost as xgb_
from catboost import CatBoostRegressor

def _ratio_fold(f):
    tr, va = f["train"], f["val"]
    ytr = np.log1p(y.loc[tr]) - np.log1p(r7.loc[tr])
    yva = np.log1p(y.loc[va]) - np.log1p(r7.loc[va])
    k = ytr.notna()
    return tr, va, ytr[k], yva, k

def gbm3(fit_predict):
    X = matrix()
    out = []
    for f in SEL:
        tr, va, ytr, yva, k = _ratio_fold(f)
        raw = fit_predict(X.loc[tr][k], ytr, X.loc[va], yva)
        p = np.expm1(pd.Series(raw, index=va) + np.log1p(r7.loc[va]))
        out.append((mae(y.loc[va], p), smape(y.loc[va], p)))
    return out

def fp_xgb(Xtr, ytr, Xva, yva):
    m = xgb_.XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=4, min_child_weight=5,
                          subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=0,
                          early_stopping_rounds=50)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m.predict(Xva)

def fp_cat(Xtr, ytr, Xva, yva):
    m = CatBoostRegressor(iterations=600, learning_rate=0.05, depth=5, random_seed=42,
                          loss_function="RMSE", verbose=0, early_stopping_rounds=50)
    m.fit(Xtr, ytr, eval_set=(Xva, yva))
    return m.predict(Xva)

g3 = {"LightGBM-ratio (V1)": vres["V1 비율타깃 log(y/roll7)"],
      "XGBoost-ratio": gbm3(fp_xgb), "CatBoost-ratio": gbm3(fp_cat)}
v1m = np.mean([r[0] for r in g3["LightGBM-ratio (V1)"]])
g3t = pd.DataFrame({n: {"MAE 평균(원)": won(np.mean([r[0] for r in rs])),
                        "sMAPE 평균": f"{np.mean([r[1] for r in rs]):.1f}%",
                        "vs V1": f"{np.mean([r[0] for r in rs]) / v1m - 1:+.1%}",
                        "regime fold(2026-03) MAE(원)": won(rs[-1][0])}
                   for n, rs in g3.items()}).T
display(g3t)

### §3b 관찰 — GBM 계열 확정

- 같은 하이브리드 구조·동일 조건에서 **LightGBM 승**: XGBoost-ratio +7.9%, CatBoost-ratio +9.9%(vs V1).
- 격차의 대부분은 **regime fold(2026-03)**에서 발생 — XGB·CatBoost는 재개장 급변 구간 적응이 LGBM보다 둔함
  (CatBoost는 안정 구간 fold에선 최상위권이지만 regime fold에서 무너져 평균이 MA-7 수준).
- 순정 log 타깃에서도 CatBoost는 LGBM·XGB보다 열세(로컬 출력 참조) → **GBM 계열 선택은 LightGBM으로 종결**.
  잔여 개선 여지는 계열 교체가 아니라 하이퍼파라미터 튜닝(M6.A6에서 Optuna/FLAML로 우리 fold 하네스 위에서).

In [ ]:
# §4 승자(V1) hold 피처 ablation — 02_features §7 hold 잔여분 최종 처분
v1_mae = np.mean([r[0] for r in vres["V1 비율타깃 log(y/roll7)"]])
rows = []
for label, extra in [("+ month sin/cos", ("month_sin", "month_cos")),
                     ("+ roll7_alcohol_share", ("roll7_alcohol_share",)),
                     ("+ days_gap_prev_open", ("days_gap_prev_open",))]:
    m_ = np.mean([r[0] for r in v_run(SEL, ratio=True, extra=extra)])
    rows.append((label, won(m_), f"{m_ / v1_mae - 1:+.1%}"))
display(pd.DataFrame(rows, columns=["변형", "MAE 평균(원)", "vs V1 keep20"]))

### §4 관찰 — keep 20열 최종 확정

- hold 3종 전부 **무익 또는 유해**: month sin/cos +1.1%, 술 비중 +4.1%(regime 프록시 재확인), gap +2.3%.
- → **M6.A5 최종 피처 = keep 20열 그대로.** hold 중 is_exam만 학사일정 검수 후 재평가 대상으로 잔류
  (`AI/data/README.md` 검수 섹션). floating은 M6.A4에서 이미 제외 확정.

In [ ]:
# §5 봉인 test(2026-04) 개봉 — 사전 고정 기준(선택 fold 평균 MAE)의 승자 V1만 1회 평가 + 참조 2종
t_v1 = v_run([TEST], ratio=True)[0]
t_ma = eval_folds(pred_ma7, [TEST])[0]
t_nd = eval_folds(pred_naivedow, [TEST])[0]
ttbl = pd.DataFrame([("V1 하이브리드(승자)", won(t_v1[0]), f"{t_v1[1]:.1f}%"),
                     ("MA-7(참조)", won(t_ma[0]), f"{t_ma[1]:.1f}%"),
                     ("Naive-요일(참조)", won(t_nd[0]), f"{t_nd[1]:.1f}%")],
                    columns=["모델", "test MAE(원)", "test sMAPE"])
display(ttbl)
print(f"test에서 V1 vs MA-7: {t_v1[0]/t_ma[0]-1:+.1%} | vs Naive-요일: {t_v1[0]/t_nd[0]-1:+.1%}")

fig, ax = plt.subplots(figsize=(9, 3.6), constrained_layout=True)
months = [f["month"] for f in SEL] + [TEST["month"]]
x = np.arange(len(months))
s_v1 = [r[1] for r in vres["V1 비율타깃 log(y/roll7)"]] + [t_v1[1]]
s_ma = [r[1] for r in res["MA-7"]] + [t_ma[1]]
ax.bar(x - 0.2, s_v1, width=0.38, color=PAL["blue"], label="V1 하이브리드")
ax.bar(x + 0.2, s_ma, width=0.38, color=PAL["orange"], label="MA-7")
ax.axvline(len(SEL) - 0.5, color=PAL["ink2"], lw=1, ls="--")
ax.annotate("← 선택 | test →", (len(SEL) - 0.5, max(s_ma) * 0.97), fontsize=8.5,
            color=PAL["ink2"], ha="center")
ax.set_xticks(x, months)
ax.set_ylabel("sMAPE (%)")
ax.set_title("fold별 sMAPE — V1 하이브리드 vs MA-7 (오른쪽 끝 = 봉인 test)")
ax.legend(frameon=False, fontsize=9)
ax.grid(axis="x", visible=False)
plt.show()

### §5 관찰 — 일반화 확인

- test(2026-04)에서 V1 **sMAPE 30.0%**, MA-7 대비 MAE **-19.6%**, naive-요일 대비 **-54%** —
  선택 fold에서의 우위가 봉인 구간에서도 유지(오히려 확대).
- test sMAPE(30%)가 선택 평균(49%)보다 좋은 이유: 2026-04는 재개장 후 운영이 안정된 무휴 영업 구간
  (EDA §2.6)이라 노이즈가 낮음 — regime 안정기의 기대 성능으로 해석.

## §6 판정·다음 단계

**M6.A5 종료 판정** — 나이브 2·스무딩 1·통계 1·GBM 2 + 하이브리드 변형 4 = **10개 후보를 동일 하네스로
비교**하고, 사전 고정 기준(선택 fold 평균 MAE)으로 승자 확정 + 봉인 test 1회 검증 완료. 산출물(모델별
평가 지표 표) 충족.

### M6.A6(초기 모델 선정)에 넘기는 권고

| 항목 | 권고 | 근거 |
|---|---|---|
| 주 모델 | **LightGBM 비율 타깃 하이브리드(V1)** — `log1p(y)−log1p(roll7)` 편차 학습, 워밍업 NaN 라벨 명시 제거 | 선택 -8.8%·test -19.6% vs 차점자, regime 강건(§3) |
| 보조 baseline | **MA-7** | 구현 0비용 fallback + drift 감시 기준선(모델이 MA-7에 지기 시작하면 경보) |
| 기각 | SARIMA(결측 많은 시계열에 둔함)·XGBoost·CatBoost(같은 비율 구조에서도 +7.9%/+9.9% 열세 §3b)·순정 LGBM·블렌드 | §2·§3·§3b |
| DNN(M6.A9) | **보류 권고** — "GBM이 베이스라인 압도" 조건도 이제 겨우 충족, 표본 256일로는 DNN 이점 없음. AutoGluon-TS probe는 M6.A9에서(우리 fold 하네스로 채점) | research §2.1 전환 조건표 |
| 튜닝 | 계열 교체 대신 V1 하이퍼파라미터 튜닝(Optuna/FLAML, 선택 fold 내에서만) — M6.A6 | §3b |
| 평가 지표 확정안 | MAE(주)+sMAPE(보고). MAPE 제외(소액일 왜곡) | §1, EDA §2.4 |

- spec 갱신(M6.A6에서 docs PR): `model_spec.md` §3 초기 모델(LGBM 하이브리드 + 라이브러리 버전),
  §7 평가 지표. `feature_spec.md` §5.2 ROI 갱신 여부는 담당자 확인.
- M6.A7(XAI): TreeSHAP은 **편차 모델**에 적용 — 기여도가 곧 "평소 대비 증감 요인"이라 자연어 근거
  템플릿과 직결. M6.A8(신뢰도): fold별 sMAPE 분포(30~64%)가 신뢰도 밴드 산정의 입력.
- 재학습·창 크기(N) probe는 하네스 재사용으로 M6.A6+에서.

### 검수 연계 (변동 없음)

휴업 사유(regime 해석 확정)·학사일정 시험주간(is_exam 재평가)·유동인구 원본 — `AI/data/README.md` 검수 섹션.